# 2 — Freeze the 3,456-row manifest and initialization pairing
No prior outcomes are read. Resolve each unique reset twice, then require one fingerprint across all six horizon/delay cells in each within-scene seed block.


In [ ]:
import csv, os, subprocess
from collections import Counter
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; OUT=Path.home()/"stage3_new"; P=Path.home()/"LIBERO-plus"; N=Path.home()/"stage1-native"
ID=Path.home()/"venv-stage1-id/bin/python"; OOD=Path.home()/"venv-stage1-ood/bin/python"; MAN=OUT/"stage3_new_manifest.csv"; AUD=OUT/"stage3_new_initialization_pairing_audit.csv"
bench="anonymous-source"; plus=subprocess.run(["git","-C",str(P),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip()
subprocess.run([str(ID),"-m","async_vla_benchmark.scripts.make_stage3_new_manifest","--output",str(MAN),"--git-sha",bench,"--lerobot-git-sha","2aba372b4e217cc47db28e0f836859b20d1456c9","--libero-plus-git-sha",plus,"--model-revision","8e174154ef5f6c60a8da12ae99c303d8963138c1"],cwd=R,check=True)
base=os.environ.copy(); base.update({"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg"})
for scene,py in (("id",ID),("ood",OOD)):
    env=base.copy()
    if scene=="ood": env.update({"PYTHONPATH":str(P),"MAGICK_HOME":str(N),"PATH":str(N/"bin")+os.pathsep+env.get("PATH",""),"LD_LIBRARY_PATH":str(N/"lib")+os.pathsep+env.get("LD_LIBRARY_PATH","")})
    subprocess.run([str(py),"-m","async_vla_benchmark.scripts.resolve_stage3_initializations","--config",str(R/"async_vla_benchmark/configs/stage3_new.yaml"),"--manifest",str(MAN),"--scene",scene,"--expected-rows","3456","--expected-cells-per-key","6","--audit-output",str(AUD)],cwd=R,env=env,check=True)


In [ ]:
rows=list(csv.DictReader(open(MAN))); audit=list(csv.DictReader(open(AUD)))
assert len(rows)==3456 and len({r['run_id'] for r in rows})==3456
assert Counter(r['scene'] for r in rows)==Counter({'id':1152,'ood':2304})
assert {int(r['seed']) for r in rows}==set(range(46,110)); assert {int(r['configured_n_action_steps']) for r in rows}=={20,25,30}; assert {int(r['added_delay_ms']) for r in rows}=={0,200}
assert {r['execution_method'] for r in rows}=={'rtc'}; assert {r['candidate_key'] for r in rows if r['scene']=='ood'}=={'long_stove_object_layout','goal_robot_initial_state','goal_light_conditions','goal_sensor_noise_posthoc','spatial_object_layout','goal_object_layout'}
assert all(r['candidate_key']=='shared_id' and r['perturbation_key']=='id' for r in rows if r['scene']=='id')
assert len(audit)==576 and all(r['repeatability_pass']=='True' and r['fingerprint']==r['repeat_fingerprint'] for r in audit)
subprocess.run([str(ID),"-m","async_vla_benchmark.scripts.validate_stage3_new","--manifest",str(MAN),"--output-dir",str(OUT),"--allow-incomplete"],cwd=R,check=True)
print("PASS: frozen 3,456 physical episodes (1,152 shared ID + 2,304 OOD); 576 reset identities repeated exactly")
print("STOP HERE and preserve the manifest SHA before notebook 03.")
